# ADSP 31014 Statistical Models for Data Science — Homework 3

Place `chicago_taxi_trip_sample.csv` in the same folder as this notebook, then run all cells (Kernel → Restart & Run All).
Answers that need interpretation are written in the markdown cells; fill in/confirm them after seeing your own output.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

df = pd.read_csv('chicago_taxi_trip_sample.csv')
print(df.shape)
df.head()

In [ ]:
# Check the columns and the levels of the categorical variables
print(df[['trip_seconds', 'fare', 'hour_type', 'pickup_area']].info())
print(df['hour_type'].value_counts(), '\n')
print(df['pickup_area'].value_counts())

# Drop rows with missing values in the variables we use (no-op if there are none)
df = df.dropna(subset=['trip_seconds', 'fare', 'hour_type', 'pickup_area']).reset_index(drop=True)

# Note: statsmodels/patsy, like R, uses the alphabetically first level as the reference level.
hour_levels = sorted(df['hour_type'].unique())
area_levels = sorted(df['pickup_area'].unique())
print('\nhour_type levels :', hour_levels)
print('pickup_area levels:', area_levels)

# Convenience: find the exact label for O'Hare (apostrophe styles can differ)
OHARE = [a for a in area_levels if 'hare' in a.lower()][0]
print('O\'Hare label in data:', repr(OHARE))

---
## Question 1a — Two categorical inputs (no interaction)

In [ ]:
m1a = smf.ols('trip_seconds ~ hour_type + pickup_area', data=df).fit()
print(m1a.summary())

In [ ]:
print('R^2                 :', round(m1a.rsquared, 4))
print('Number of beta params:', len(m1a.params))
print()
print(m1a.params.round(4))

In [ ]:
# Fitted equation written out in mathematical form
def equation(params, response='y'):
    terms = []
    for name, b in params.items():
        if name == 'Intercept':
            terms.append(f'{b:.4f}')
            continue
        # patsy names look like  hour_type[T.Rush Hour]  or  pickup_area[T.Downtown]
        var, lvl = name.split('[T.')
        lvl = lvl.rstrip(']')
        terms.append(f'{b:+.4f} * 1{{{var}={lvl}}}')
    return f'{response} = ' + ' '.join(terms)

print(equation(m1a.params))

In [ ]:
new = pd.DataFrame({'hour_type': ['Rush Hour'], 'pickup_area': ['Downtown']})
print('Predicted trip_seconds (Rush Hour, Downtown):', m1a.predict(new).iloc[0])

**Answer 1a**

- R² = *(see output above)*
- Number of β's = 1 (intercept) + (#hour_type levels − 1) + (#pickup_area levels − 1) *(see output above)*
- The fitted equation is printed above. The reference levels (hour_type and pickup_area, alphabetically first) are absorbed in the intercept β₀*.
- Predicted ŷ for Rush Hour & Downtown = *(see output above)*; it equals β₀* + β(hour_type=Rush Hour) + β(pickup_area=Downtown), with a term dropped if that level is the reference level.

---
## Question 1b — Categorical interaction

In [ ]:
m1b = smf.ols('trip_seconds ~ hour_type * pickup_area', data=df).fit()
print(m1b.summary())
print('R^2                 :', round(m1b.rsquared, 4))
print('Number of beta params:', len(m1b.params))

In [ ]:
new = pd.DataFrame({'hour_type': ['Rush Hour'], 'pickup_area': [OHARE]})
print("Predicted trip_seconds (Rush Hour, O'Hare):", m1b.predict(new).iloc[0])

In [ ]:
# Sample mean of trip_seconds for each (hour_type, pickup_area) combination
grp = (df.groupby(['hour_type', 'pickup_area'])['trip_seconds']
         .agg(['mean', 'count']).reset_index())
print(grp)

print()
print('Cross-tab of means:')
print(df.pivot_table(index='hour_type', columns='pickup_area', values='trip_seconds', aggfunc='mean').round(2))

imin = grp['mean'].idxmin()
print('\nSmallest sample mean:')
print(grp.loc[imin])

In [ ]:
# Relationship between the sample means and the fitted coefficients:
# The prediction for each cell of the model equals that cell's sample mean.
cells = grp[['hour_type', 'pickup_area']].copy()
cells['fitted'] = m1b.predict(cells)
cmp = cells.merge(grp, on=['hour_type', 'pickup_area'])
cmp['abs_diff'] = (cmp['fitted'] - cmp['mean']).abs()
print(cmp)
print('Max |fitted - sample mean|:', cmp['abs_diff'].max())

# Reconstruct a cell mean by summing the relevant coefficients, e.g. Rush Hour & O'Hare
p = m1b.params
print(p.round(4))
names = [n for n in p.index if n == 'Intercept'
         or (n.startswith('hour_type[T.Rush Hour]') and n.endswith(f'pickup_area[T.{OHARE}]'))
         or n == 'hour_type[T.Rush Hour]' or n == f'pickup_area[T.{OHARE}]']
print('\nCoefficients summed for (Rush Hour, O\'Hare):', names)
print('Sum =', p[names].sum())

**Answer 1b**

- R², number of β's, the prediction for (Rush Hour, O'Hare), and the smallest sample mean are shown in the outputs above.
- **Relationship between sample means and coefficients:** with two categorical variables and a full interaction, the model has one free parameter for every cell (combination). The fitted value for each cell is exactly the sample mean of `trip_seconds` in that cell (the check above shows the difference is ~0). The intercept β₀* is the sample mean of the reference cell, and each other cell's mean is recovered by adding the intercept, the matching main-effect coefficients, and the matching interaction coefficient. So the coefficients are just differences/contrasts between the cell sample means.
- **Why the number of β's makes sense:** let H = number of `hour_type` levels and A = number of `pickup_area` levels. The model has 1 + (H−1) + (A−1) + (H−1)(A−1) = H·A parameters, i.e. one parameter per combination of levels, which is what is needed to give each of the H·A cells its own mean.

---
## Question 1c — Include `fare`

In [ ]:
# patsy supports this formula as written. It is the same as hour_type * pickup_area * fare.
f1c = 'trip_seconds ~ hour_type * pickup_area + fare:(hour_type * pickup_area)'
m1c = smf.ols(f1c, data=df).fit()
print(m1c.summary())
print('R^2                 :', round(m1c.rsquared, 4))
print('Number of beta params:', len(m1c.params))

**Number of β's:** the model fits a separate straight line (intercept + slope = 2 parameters) for each of the H·A combinations, so it has 2·H·A parameters. This is the H·A parameters of the cell-wise intercepts (as in Question 1b) plus H·A additional parameters for the cell-wise `fare` slopes. Compare with the printed count above.

In [ ]:
# Line implied by the model for (Rush Hour, O'Hare):  y = a0 + a1 * fare
p = m1c.params
print(p.round(4))

# (i) Add up the relevant coefficients by hand
def has_level(name, var, lvl):
    return f'{var}[T.{lvl}]' in name

rel = []
for n in p.index:
    parts = n.split(':')
    # which categorical terms does this coefficient involve?
    ok = True
    for part in parts:
        if part.startswith('hour_type[') and not has_level(part, 'hour_type', 'Rush Hour'): ok = False
        if part.startswith('pickup_area[') and not has_level(part, 'pickup_area', OHARE): ok = False
    # drop coefficients that involve the *other* hour_type / area levels; keep intercept & fare
    if ok:
        rel.append(n)
# reference levels have no dummy column, so check against the full set carefully
a0_names = [n for n in rel if 'fare' not in n]
a1_names = [n for n in rel if 'fare' in n]
print('\nIntercept terms used:', a0_names)
print('Slope terms used    :', a1_names)
a0_manual = p[a0_names].sum()
a1_manual = p[a1_names].sum()
print(f'a0 (by summing coefficients) = {a0_manual:.4f}')
print(f'a1 (by summing coefficients) = {a1_manual:.4f}')

# (ii) Cross-check using the model's own predictions at fare = 0 and fare = 1
nd = pd.DataFrame({'hour_type': ['Rush Hour']*2, 'pickup_area': [OHARE]*2, 'fare': [0.0, 1.0]})
yhat = m1c.predict(nd).values
print(f'\na0 (prediction at fare=0)    = {yhat[0]:.4f}')
print(f'a1 (pred(1) - pred(0))       = {yhat[1]-yhat[0]:.4f}')

In [ ]:
# Verification: fit a separate simple regression on the (Rush Hour, O'Hare) subset only
sub = df[(df['hour_type'] == 'Rush Hour') & (df['pickup_area'] == OHARE)]
m_sub = smf.ols('trip_seconds ~ fare', data=sub).fit()
print('n rows in subset:', len(sub))
print(m_sub.summary().tables[1])
print(f'\nSubset fit:  a0 = {m_sub.params["Intercept"]:.4f},  a1 = {m_sub.params["fare"]:.4f}')

**Answer 1c**

- R², β count, and a₀, a₁ for (Rush Hour, O'Hare) are shown above. a₀ is the intercept plus the relevant main-effect and interaction intercept shifts; a₁ is the `fare` coefficient plus the relevant `fare:` interaction shifts (when the level is not the reference level).
- The separate subset regression gives the **same** a₀ and a₁ (up to numerical precision), because a model with all interactions between `fare` and the cell indicators is equivalent to fitting each cell separately. The only difference is that the full model uses one pooled residual variance, so standard errors differ slightly, but the point estimates are identical.

---
## Question 1d — Three-way interaction

In [ ]:
m1d = smf.ols('trip_seconds ~ hour_type * pickup_area * fare', data=df).fit()
print(m1d.summary())
print('R^2                 :', round(m1d.rsquared, 6), ' (1c:', round(m1c.rsquared, 6), ')')
print('Number of beta params:', len(m1d.params), ' (1c:', len(m1c.params), ')')
print('Same fitted values?  ', np.allclose(m1d.fittedvalues, m1c.fittedvalues))
print('Same residual SS?    ', np.isclose(m1d.ssr, m1c.ssr))

# Compare coefficients (after sorting the terms into a canonical order)
def canon(name):
    return ':'.join(sorted(name.split(':')))
c1 = m1c.params.rename(index=canon).sort_index()
c2 = m1d.params.rename(index=canon).sort_index()
print('Same coefficients (matched by term)?', np.allclose(c1.values, c2.values))

**Answer 1d**

`hour_type * pickup_area * fare` expands to all main effects, all two-way interactions and the three-way interaction `hour_type:pickup_area:fare`, which is exactly the same set of terms as `hour_type * pickup_area + fare:(hour_type * pickup_area)`. So the two formulas describe the **same model**: identical number of coefficients, identical coefficient estimates (only the order of terms in the label may differ), identical R², and identical fitted values. The `*` form is just a shorter way to write it.

---
## Question 2 — Asking GenAI about categorical variables with many levels

*Replace or extend this with the response from the GenAI tool you actually asked. Here is a summary of what Claude says:*

**Prompt used:** "How should I handle a categorical variable with many levels when fitting a linear regression model?"

**Summary of the answer:**

1. **Why it is a problem.** A categorical variable with K levels needs K−1 dummy variables. With many levels this inflates the number of parameters, so the model overfits, standard errors become large, some levels have very few observations (unstable or unestimable coefficients), and new/unseen levels at prediction time cannot be handled.
2. **Collapse/group levels.** Merge rare levels into an "Other" category, or group levels using domain knowledge (e.g. ZIP codes → region, city → state).
3. **Use a numeric summary of the level (encoding).** Replace the category with something meaningful such as frequency/count encoding or target (mean) encoding. Target encoding must be computed within cross-validation folds to avoid leakage.
4. **Regularization.** Fit ridge/lasso/elastic net on the dummy variables so that coefficients of weakly supported levels are shrunk (lasso can set them to zero).
5. **Mixed-effects / hierarchical models.** Treat the variable as a random effect (partial pooling), so levels with little data borrow strength from the overall mean. This also handles unseen levels more gracefully.
6. **Feature selection / testing.** Use an F-test (ANOVA) for the variable as a whole instead of looking at individual dummy t-tests, and use cross-validation to choose the amount of grouping or shrinkage.
7. **Practical tips.** Choose a reference level with plenty of data, make sure each level has enough observations, and avoid high-order interactions between several high-cardinality variables, since the parameter count multiplies (as seen in Question 1b/1c).